# Scraping referendum per sezione

Questo notebook e' uno scheletro operativo per scaricare i CSV dalle pagine comunali di Eligendo.

Workflow previsto:
1. Leggere i link da `linkdownload_sezioni.txt`.
2. Aprire ogni pagina con un browser automatizzato.
3. Cercare il bottone `Esporta CSV`.
4. Salvare il file scaricato in una cartella locale.

Nota: non lo sto eseguendo qui. Va rifinito e lanciato in locale da te.

## Dipendenze suggerite

Qui la base e' Selenium con Firefox:

```bash
pip install pandas selenium jupyter
```

Serve anche Firefox installato nel sistema. Se `geckodriver` non e' nel PATH, dovrai indicarlo a mano nella funzione `build_firefox_driver`.

In [50]:
import time

In [51]:
print("start at:" +time.ctime())
start = time.ctime()

start at:Mon Mar 23 21:41:34 2026


In [52]:
from __future__ import annotations

import json
import re
import time
from pathlib import Path
from typing import Dict

import pandas as pd

from selenium import webdriver
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.firefox.options import Options as FirefoxOptions
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

ROOT = Path.cwd()
LINKS_FILE = ROOT / "linkdownload_sezioni.txt"
DOWNLOAD_TURNOUT_DIR = ROOT / "downloads_csv"
DOWNLOAD_RESULTS_DIR = ROOT / "downloads_scrutini"
RESULTS_SECTIONS_DIR = ROOT / "scrutini_sezioni"
DOWNLOAD_TURNOUT_DIR.mkdir(exist_ok=True)
DOWNLOAD_RESULTS_DIR.mkdir(exist_ok=True)
RESULTS_SECTIONS_DIR.mkdir(exist_ok=True)

DATE_TAG = "2026-03-22"


In [53]:
def parse_links_file(path: Path) -> Dict[str, str]:
    """Parsa il file testuale tipo:
    "roma":"https://...",
    "milano":"https://..."
    """
    text = path.read_text(encoding="utf-8")
    pairs = re.findall(r'"([^"]+)"\s*:\s*"([^"]+)"', text)
    if not pairs:
        raise ValueError(f"Nessun link trovato in {path}")
    return dict(pairs)


city_links = parse_links_file(LINKS_FILE)
city_links_scrutini = {city: url.replace("/votanti/", "/scrutini/") for city, url in city_links.items()}
pd.DataFrame(
    [
        {"city": city, "turnout_url": url, "results_url": city_links_scrutini[city]}
        for city, url in city_links.items()
    ]
)


,city,turnout_url,results_url
0,roma,https://elezioni.interno.gov.it/risultati/2026...,https://elezioni.interno.gov.it/risultati/2026...
1,milano,https://elezioni.interno.gov.it/risultati/2026...,https://elezioni.interno.gov.it/risultati/2026...
2,napoli,https://elezioni.interno.gov.it/risultati/2026...,https://elezioni.interno.gov.it/risultati/2026...
3,bologna,https://elezioni.interno.gov.it/risultati/2026...,https://elezioni.interno.gov.it/risultati/2026...
4,torino,https://elezioni.interno.gov.it/risultati/2026...,https://elezioni.interno.gov.it/risultati/2026...
5,genova,https://elezioni.interno.gov.it/risultati/2026...,https://elezioni.interno.gov.it/risultati/2026...
6,firenze,https://elezioni.interno.gov.it/risultati/2026...,https://elezioni.interno.gov.it/risultati/2026...
7,palermo,https://elezioni.interno.gov.it/risultati/2026...,https://elezioni.interno.gov.it/risultati/2026...
8,reggiocalabria,https://elezioni.interno.gov.it/risultati/2026...,https://elezioni.interno.gov.it/risultati/2026...
9,taranto,https://elezioni.interno.gov.it/risultati/2026...,https://elezioni.interno.gov.it/risultati/2026...


## Test rapido su una singola citta'

Questa funzione prova a:
- aprire la pagina;
- aspettare il rendering;
- trovare un elemento con testo `Esporta CSV`;
- scaricare il file.

Se il sito cambia struttura, i selettori da ritoccare saranno soprattutto dentro `candidate_xpaths`.

In [54]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

In [55]:
# def slugify(value: str) -> str:
#     return re.sub(r"[^a-z0-9]+", "_", value.lower()).strip("_")


# def build_firefox_driver(download_dir: Path, headless: bool = True) -> webdriver.Firefox:
#     options = FirefoxOptions()
#     if headless:
#         options.add_argument("-headless")

#     options.set_preference("browser.download.folderList", 2)
#     options.set_preference("browser.download.dir", str(download_dir.resolve()))
#     options.set_preference("browser.download.useDownloadDir", True)
#     options.set_preference("browser.helperApps.neverAsk.saveToDisk", "text/csv,application/csv,application/octet-stream")
#     options.set_preference("browser.download.manager.showWhenStarting", False)
#     options.set_preference("pdfjs.disabled", True)
#     options.set_preference("browser.download.alwaysOpenPanel", False)

#     driver = webdriver.Firefox(options=options)
#     driver.set_window_size(1600, 2200)
#     try:
#         driver.maximize_window()
#     except Exception:
#         pass
#     return driver


# def wait_for_new_csv(download_dir: Path, existing_files: set[Path], timeout_s: int = 60) -> Path:
#     wait = WebDriverWait(object(), timeout_s, poll_frequency=1)

#     def _download_completed(_driver_placeholder):
#         current_files = set(download_dir.glob("*.csv"))
#         new_files = sorted(current_files - existing_files, key=lambda p: p.stat().st_mtime)
#         partial_files = list(download_dir.glob("*.part"))
#         if new_files and not partial_files:
#             return new_files[-1]
#         return False

#     return wait.until(_download_completed)


# def download_city_csv(
#     city: str,
#     url: str,
#     download_dir: Path = DOWNLOAD_TURNOUT_DIR,
#     headless: bool = True,
#     timeout_s: int = 45,
# ) -> Path:
#     destination = download_dir / f"{DATE_TAG}_{slugify(city)}.csv"
#     temp_destination = download_dir / f"{DATE_TAG}_{slugify(city)}__new.csv"
#     driver = build_firefox_driver(download_dir, headless=headless)

#     try:
#         wait = WebDriverWait(driver, timeout_s)
#         existing_files = set(download_dir.glob("*.csv"))

#         driver.get(url)
#         wait.until(lambda d: d.execute_script("return document.readyState") == "complete")
#         time.sleep(7)

#         candidate_xpaths = [
#             "//*[contains(concat(' ' , normalize-space(@class), ' ' ), ' q-btn__content ' ) and contains(concat(' ' , normalize-space(@class), ' ' ), ' text-center ' ) and contains(concat(' ' , normalize-space(@class), ' ' ), ' q-anchor--skip ' ) and contains(translate(normalize-space(.), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'esporta in csv')]",
#             "//button//*[contains(concat(' ' , normalize-space(@class), ' ' ), ' q-btn__content ' ) and contains(translate(normalize-space(.), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'esporta in csv')]",
#             "//a//*[contains(concat(' ' , normalize-space(@class), ' ' ), ' q-btn__content ' ) and contains(translate(normalize-space(.), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'esporta in csv')]",
#             "//*[contains(translate(normalize-space(.), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'esporta in csv')]",
#         ]

#         trigger = None
#         time.sleep(4)
#         for xpath in candidate_xpaths:
#             matches = driver.find_elements(By.XPATH, xpath)
#             visible_matches = [element for element in matches if element.is_displayed()]
#             if visible_matches:
#                 trigger = visible_matches[0]
#                 break

#         if trigger is None:
#             driver.save_screenshot(str(download_dir / f"debug_{slugify(city)}.png"))
#             raise RuntimeError(f"Bottone 'Esporta CSV' non trovato per {city}")

#         time.sleep(2)
#         wait.until(EC.element_to_be_clickable(trigger))
#         time.sleep(1)
#         trigger.click()

#         downloaded_file = wait_for_new_csv(download_dir, existing_files, timeout_s=60)
#         if temp_destination.exists():
#             temp_destination.unlink()
#         downloaded_file.replace(temp_destination)
#         temp_destination.replace(destination)
#         return destination

#     except TimeoutException as exc:
#         driver.save_screenshot(str(download_dir / f"timeout_{slugify(city)}.png"))
#         raise RuntimeError(f"Timeout su {city}: {url}") from exc

#     finally:
#         driver.quit()


In [56]:
# Esempio rapido: affluenza
# path_roma = download_city_csv('roma', city_links['roma'], download_dir=DOWNLOAD_TURNOUT_DIR, headless=False)
# path_roma

# Gli scrutini vengono gestiti in notebook separati e salvati in `scrutini_sezioni/`.


## Batch su tutte le citta'

Questa cella salva i risultati e continua anche se una citta' fallisce.

In [57]:
# download_results = []

# for city, url in city_links.items():
#     try:
#         saved_path = download_city_csv(city, url, download_dir=DOWNLOAD_TURNOUT_DIR, headless=True)
#         download_results.append({
#             'job': 'affluenza',
#             'city': city,
#             'url': url,
#             'status': 'ok',
#             'file': str(saved_path),
#         })
#     except Exception as exc:
#         download_results.append({
#             'job': 'affluenza',
#             'city': city,
#             'url': url,
#             'status': 'error',
#             'file': None,
#             'error': str(exc),
#         })

# download_results_df = pd.DataFrame(download_results)
# download_results_df


## Controllo veloce dei CSV scaricati

Quando avrai i file, questa cella aiuta a ispezionare intestazioni e prime righe sia per l'affluenza sia per gli scrutini.


In [58]:
# Scegli qui quale cartella ispezionare.
inspection_dir = RESULTS_SECTIONS_DIR  # oppure DOWNLOAD_RESULTS_DIR o DOWNLOAD_TURNOUT_DIR
csv_files = sorted(inspection_dir.glob('*.csv'))
csv_files[:10]


[WindowsPath('c:/Users/gabri/Documents/GitHub/referendum2026_mappepersezione/scrutini_sezioni/bologna.csv'),
 WindowsPath('c:/Users/gabri/Documents/GitHub/referendum2026_mappepersezione/scrutini_sezioni/firenze.csv'),
 WindowsPath('c:/Users/gabri/Documents/GitHub/referendum2026_mappepersezione/scrutini_sezioni/milano.csv'),
 WindowsPath('c:/Users/gabri/Documents/GitHub/referendum2026_mappepersezione/scrutini_sezioni/napoli.csv'),
 WindowsPath('c:/Users/gabri/Documents/GitHub/referendum2026_mappepersezione/scrutini_sezioni/palermo.csv'),
 WindowsPath('c:/Users/gabri/Documents/GitHub/referendum2026_mappepersezione/scrutini_sezioni/reggiocalabria.csv'),
 WindowsPath('c:/Users/gabri/Documents/GitHub/referendum2026_mappepersezione/scrutini_sezioni/roma.csv'),
 WindowsPath('c:/Users/gabri/Documents/GitHub/referendum2026_mappepersezione/scrutini_sezioni/taranto.csv'),
 WindowsPath('c:/Users/gabri/Documents/GitHub/referendum2026_mappepersezione/scrutini_sezioni/torino.csv')]

In [59]:
# if csv_files:
#     sample_df = pd.read_csv(csv_files[0], sep=';', encoding='utf-8-sig')
#     display(sample_df.head())
#     print(sample_df.columns.tolist())
# else:
#     print('Nessun CSV disponibile nella cartella selezionata.')


## Esporta JSON per il sito

Questa cella converte i CSV di affluenza e di scrutinio in file JSON leggeri, pronti per il frontend.


In [60]:
DATA_DIR = ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)

SLOT_ORDER = ['sun_12', 'sun_19', 'sun_23', 'mon_12', 'mon_15']
import unicodedata

DAY_PREFIX = {'domenica': 'sun', 'lunedi': 'mon'}

def clean_text(value: str | None) -> str:
    if value is None:
        return ''
    return str(value).strip().strip('\"')

def normalize_header(value: str | None) -> str:
    value = clean_text(value).lower()
    value = unicodedata.normalize('NFKD', value).encode('ascii', 'ignore').decode('ascii')
    value = value.replace('?\xa0', ' ').replace('\xa0', ' ')
    value = re.sub(r'\s+', ' ', value)
    return value.strip()

def parse_numeric(value: str | None) -> float | None:
    if value is None:
        return None
    value = clean_text(value).replace('%', '').replace('?\xa0', ' ').replace('\xa0', ' ').strip()
    if not value:
        return None

    missing_markers = {
        'n.p.', 'np', 'n.p',
        'n.d.', 'nd', 'n.d',
        'non pervenuto', 'non disponibile',
        '-', '--', 's.t.', 's.t'
    }
    normalized_value = normalize_header(value).replace(' ', '')
    normalized_missing = {normalize_header(marker).replace(' ', '') for marker in missing_markers}
    if normalized_value in normalized_missing:
        return None

    if ',' in value and '.' in value:
        if value.rfind(',') > value.rfind('.'):
            value = value.replace('.', '').replace(',', '.')
        else:
            value = value.replace(',', '')
    elif ',' in value:
        value = value.replace('.', '').replace(',', '.')
    elif re.match(r'^\d{1,3}(\.\d{3})+$', value):
        value = value.replace('.', '')
    return float(value)

def normalize_day_label(label: str | None) -> str | None:
    if not label:
        return None
    return DAY_PREFIX.get(normalize_header(label))

def infer_turnout_updated_at(turnout_slots: list[str]) -> str:
    if 'mon_15' in turnout_slots:
        return '2026-03-23T15:00:00+01:00'
    if 'mon_12' in turnout_slots:
        return '2026-03-23T12:00:00+01:00'
    if 'sun_23' in turnout_slots:
        return '2026-03-22T23:00:00+01:00'
    if 'sun_19' in turnout_slots:
        return '2026-03-22T19:00:00+01:00'
    if 'sun_12' in turnout_slots:
        return '2026-03-22T12:00:00+01:00'
    return '2026-03-23T00:00:00+01:00'

def infer_results_updated_at(sections: list[dict]) -> str:
    timestamps = []
    for row in sections:
        updated_at = row.get('updated_at')
        if updated_at:
            timestamps.append(updated_at)
    return max(timestamps) if timestamps else '2026-03-23T00:00:00+01:00'

def parse_update_timestamp(value: str | None) -> str | None:
    value = clean_text(value)
    if not value:
        return None
    match = re.match(r'^(\d{2})/(\d{2})/(\d{4})\s*-\s*(\d{2}):(\d{2})$', value)
    if not match:
        return None
    day, month, year, hour, minute = match.groups()
    return f'{year}-{month}-{day}T{hour}:{minute}:00+01:00'

def split_semicolon_line(line: str) -> list[str]:
    return [clean_text(item) for item in line.split(';')]

def derive_turnout_columns(day_cells: list[str], headers: list[str]) -> list[tuple[str, int]]:
    turnout_candidates = []
    for idx, header in enumerate(headers):
        match = re.match(r'^% ore (\d+)$', clean_text(header), flags=re.I)
        if match:
            turnout_candidates.append((idx, match.group(1)))

    if not turnout_candidates:
        return []

    raw_days = [clean_text(day_cells[idx]) if idx < len(day_cells) else '' for idx, _ in turnout_candidates]
    first_non_empty = next((value for value in raw_days if value), '')
    current_day = normalize_day_label(first_non_empty)
    normalized_days = []
    for raw_day in raw_days:
        if raw_day:
            current_day = normalize_day_label(raw_day)
        normalized_days.append(current_day)

    turnout_columns = []
    legacy_map = {'12': 'sun_12', '19': 'sun_19', '23': 'sun_23', '15': 'mon_15'}
    for (idx, hour), day_key in zip(turnout_candidates, normalized_days):
        if day_key:
            turnout_columns.append((f'{day_key}_{hour}', idx))
        elif hour in legacy_map:
            turnout_columns.append((legacy_map[hour], idx))
    return turnout_columns

def parse_turnout_csv(csv_path: Path) -> tuple[list[dict], list[str]]:
    rows = []
    lines = [line.strip() for line in csv_path.read_text(encoding='utf-8-sig').splitlines() if line.strip()]
    if len(lines) < 3:
        return rows, []

    day_cells = split_semicolon_line(lines[0])
    headers = split_semicolon_line(lines[1])
    ente_idx = headers.index('Ente')
    turnout_columns = derive_turnout_columns(day_cells, headers)

    for line in lines[3:]:
        parts = split_semicolon_line(line)
        if len(parts) <= ente_idx:
            continue
        name = parts[ente_idx]
        match = re.match(r'^SEZIONE\s+(\d+)$', name, flags=re.I)
        if not match:
            continue

        turnout = {slot: None for slot, _ in turnout_columns}
        for slot, idx in turnout_columns:
            turnout[slot] = parse_numeric(parts[idx]) if idx < len(parts) else None

        rows.append({'section': int(match.group(1)), 'name': name, 'turnout': turnout, 'results': None})

    turnout_slots = [slot for slot in SLOT_ORDER if any(slot in row.get('turnout', {}) for row in rows)]
    return rows, turnout_slots

def find_header_index(headers: list[str], candidates: list[str]) -> int | None:
    normalized_headers = [normalize_header(header) for header in headers]
    normalized_candidates = [normalize_header(candidate) for candidate in candidates]

    for candidate in normalized_candidates:
        for idx, header in enumerate(normalized_headers):
            if header == candidate:
                return idx

    for candidate in normalized_candidates:
        for idx, header in enumerate(normalized_headers):
            if candidate in header:
                return idx
    return None

def parse_results_csv(csv_path: Path) -> list[dict]:
    rows = []
    lines = [line.strip() for line in csv_path.read_text(encoding='utf-8-sig').splitlines() if line.strip()]
    if len(lines) < 2:
        return rows

    # Nuovo formato scraper per sezione: CSV con intestazioni semplici
    comma_headers = [normalize_header(part) for part in lines[0].split(',')]
    if 'sezione' in comma_headers and 'affluenza' in comma_headers:
        frame = pd.read_csv(csv_path, encoding='utf-8-sig')
        frame.columns = [normalize_header(column) for column in frame.columns]

        yes_col = next((column for column in frame.columns if column in {'si', 'si_pct', 'voti_si', 'perc_si', 'si_percentuale'}), None)
        no_col = next((column for column in frame.columns if column in {'no', 'no_pct', 'voti_no', 'perc_no', 'no_percentuale'}), None)

        for _, row in frame.iterrows():
            section = parse_numeric(row.get('sezione'))
            if section is None:
                continue

            yes_value = parse_numeric(row.get(yes_col)) if yes_col else None
            no_value = parse_numeric(row.get(no_col)) if no_col else None
            yes_pct = None
            no_pct = None
            if yes_col and 'pct' in yes_col or yes_col and 'perc' in yes_col or yes_col and 'percentuale' in yes_col:
                yes_pct = yes_value
            if no_col and 'pct' in no_col or no_col and 'perc' in no_col or no_col and 'percentuale' in no_col:
                no_pct = no_value
            if yes_pct is None or no_pct is None:
                if yes_value is not None and no_value is not None and (yes_value + no_value) > 0:
                    yes_pct = yes_value / (yes_value + no_value) * 100
                    no_pct = no_value / (yes_value + no_value) * 100

            rows.append({
                'section': int(section),
                'name': f'SEZIONE {int(section)}',
                'updated_at': parse_update_timestamp(row.get('aggiornamento')),
                'results': {
                    'yes': yes_value,
                    'no': no_value,
                    'yes_pct': yes_pct,
                    'no_pct': no_pct,
                    'blank': parse_numeric(row.get('schede_bianche')),
                    'null': parse_numeric(row.get('schede_nulle')),
                    'turnout': parse_numeric(row.get('affluenza')),
                },
            })
        return rows

    # Vecchio formato Eligendo esportato dal sito
    header_idx = next((idx for idx, line in enumerate(lines) if 'Ente' in line), None)
    if header_idx is None:
        return rows

    headers = split_semicolon_line(lines[header_idx])
    data_lines = lines[header_idx + 1:]
    ente_idx = headers.index('Ente')

    si_idx = find_header_index(headers, ['% si', 'si', 'si', 'voti si', '% voti si'])
    no_idx = find_header_index(headers, ['% no', 'no', 'voti no', '% voti no'])
    blank_idx = find_header_index(headers, ['schede bianche', '% schede bianche', 'bianche'])
    null_idx = find_header_index(headers, ['schede nulle', '% schede nulle', 'nulle'])
    turnout_idx = find_header_index(headers, ['affluenza', '% affluenza', '% votanti', 'votanti'])

    if si_idx is None and no_idx is None and blank_idx is None and null_idx is None and turnout_idx is None:
        return rows

    yes_header = normalize_header(headers[si_idx]) if si_idx is not None else ''
    no_header = normalize_header(headers[no_idx]) if no_idx is not None else ''
    yes_is_pct = '%' in yes_header
    no_is_pct = '%' in no_header

    for line in data_lines:
        parts = split_semicolon_line(line)
        if len(parts) <= ente_idx:
            continue
        name = parts[ente_idx]
        match = re.match(r'^SEZIONE\s+(\d+)$', name, flags=re.I)
        if not match:
            continue

        yes_value = parse_numeric(parts[si_idx]) if si_idx is not None and si_idx < len(parts) else None
        no_value = parse_numeric(parts[no_idx]) if no_idx is not None and no_idx < len(parts) else None
        blank_value = parse_numeric(parts[blank_idx]) if blank_idx is not None and blank_idx < len(parts) else None
        null_value = parse_numeric(parts[null_idx]) if null_idx is not None and null_idx < len(parts) else None
        turnout_value = parse_numeric(parts[turnout_idx]) if turnout_idx is not None and turnout_idx < len(parts) else None

        yes_pct = yes_value if yes_is_pct else None
        no_pct = no_value if no_is_pct else None
        if yes_pct is None or no_pct is None:
            if yes_value is not None and no_value is not None and (yes_value + no_value) > 0:
                yes_pct = yes_value / (yes_value + no_value) * 100
                no_pct = no_value / (yes_value + no_value) * 100

        rows.append({
            'section': int(match.group(1)),
            'name': name,
            'updated_at': None,
            'results': {
                'yes': yes_value,
                'no': no_value,
                'yes_pct': yes_pct,
                'no_pct': no_pct,
                'blank': blank_value,
                'null': null_value,
                'turnout': turnout_value,
            },
        })

    return rows

turnout_exported = []
for city in city_links:
    csv_path = DOWNLOAD_TURNOUT_DIR / f'{DATE_TAG}_{slugify(city)}.csv'
    if not csv_path.exists():
        continue
    sections, turnout_slots = parse_turnout_csv(csv_path)
    payload = {
        'city': city,
        'updated_at': infer_turnout_updated_at(turnout_slots),
        'turnout_slots': turnout_slots,
        'sections': sections,
    }
    json_path = DATA_DIR / f'turnout_{slugify(city)}.json'
    json_path.write_text(json.dumps(payload, ensure_ascii=False), encoding='utf-8')
    turnout_exported.append({'city': city, 'json': str(json_path), 'rows': len(sections), 'slots': ', '.join(turnout_slots)})

results_exported = []
results_sources = []
for city in city_links_scrutini:
    slug = slugify(city)
    section_csv = RESULTS_SECTIONS_DIR / f'{slug}.csv'
    legacy_csv = DOWNLOAD_RESULTS_DIR / f'{DATE_TAG}_{slug}.csv'
    csv_path = section_csv if section_csv.exists() else legacy_csv
    if not csv_path.exists():
        continue
    sections = parse_results_csv(csv_path)
    payload = {
        'city': city,
        'updated_at': infer_results_updated_at(sections),
        'sections': sections,
    }
    json_path = DATA_DIR / f'results_{slug}.json'
    json_path.write_text(json.dumps(payload, ensure_ascii=False), encoding='utf-8')
    results_exported.append({'city': city, 'json': str(json_path), 'rows': len(sections)})
    results_sources.append({'city': city, 'source': str(csv_path)})

print('Turnout JSON:')
display(pd.DataFrame(turnout_exported))
print('Results JSON:')
display(pd.DataFrame(results_exported))
print('Results sources:')
pd.DataFrame(results_sources)


Turnout JSON:


,city,json,rows,slots
0,roma,c:\Users\gabri\Documents\GitHub\referendum2026...,2569,"sun_12, sun_19, sun_23, mon_15"
1,milano,c:\Users\gabri\Documents\GitHub\referendum2026...,1225,"sun_12, sun_19, sun_23, mon_15"
2,napoli,c:\Users\gabri\Documents\GitHub\referendum2026...,869,"sun_12, sun_19, sun_23, mon_15"
3,bologna,c:\Users\gabri\Documents\GitHub\referendum2026...,437,"sun_12, sun_19, sun_23, mon_15"
4,torino,c:\Users\gabri\Documents\GitHub\referendum2026...,911,"sun_12, sun_19, sun_23, mon_15"
5,genova,c:\Users\gabri\Documents\GitHub\referendum2026...,646,"sun_12, sun_19, sun_23, mon_15"
6,firenze,c:\Users\gabri\Documents\GitHub\referendum2026...,354,"sun_12, sun_19, sun_23, mon_15"
7,palermo,c:\Users\gabri\Documents\GitHub\referendum2026...,593,"sun_12, sun_19, sun_23, mon_15"
8,reggiocalabria,c:\Users\gabri\Documents\GitHub\referendum2026...,192,"sun_12, sun_19, sun_23, mon_15"
9,taranto,c:\Users\gabri\Documents\GitHub\referendum2026...,189,"sun_12, sun_19, sun_23, mon_15"


Results JSON:


,city,json,rows
0,roma,c:\Users\gabri\Documents\GitHub\referendum2026...,1168
1,milano,c:\Users\gabri\Documents\GitHub\referendum2026...,779
2,napoli,c:\Users\gabri\Documents\GitHub\referendum2026...,474
3,bologna,c:\Users\gabri\Documents\GitHub\referendum2026...,397
4,torino,c:\Users\gabri\Documents\GitHub\referendum2026...,832
5,firenze,c:\Users\gabri\Documents\GitHub\referendum2026...,347
6,palermo,c:\Users\gabri\Documents\GitHub\referendum2026...,591
7,reggiocalabria,c:\Users\gabri\Documents\GitHub\referendum2026...,144
8,taranto,c:\Users\gabri\Documents\GitHub\referendum2026...,92


Results sources:


,city,source
0,roma,c:\Users\gabri\Documents\GitHub\referendum2026...
1,milano,c:\Users\gabri\Documents\GitHub\referendum2026...
2,napoli,c:\Users\gabri\Documents\GitHub\referendum2026...
3,bologna,c:\Users\gabri\Documents\GitHub\referendum2026...
4,torino,c:\Users\gabri\Documents\GitHub\referendum2026...
5,firenze,c:\Users\gabri\Documents\GitHub\referendum2026...
6,palermo,c:\Users\gabri\Documents\GitHub\referendum2026...
7,reggiocalabria,c:\Users\gabri\Documents\GitHub\referendum2026...
8,taranto,c:\Users\gabri\Documents\GitHub\referendum2026...


In [61]:
import subprocess
from pathlib import Path

def build_city_bundles(root_path):
    root = Path(root_path)
    script = root / "build_city_bundles.ps1"

    result = subprocess.run(
        [
            "powershell",
            "-ExecutionPolicy",
            "Bypass",
            "-File",
            str(script),
        ],
        cwd=root,
        text=True,
        capture_output=True,
    )

    print(result.stdout)
    if result.stderr:
        print(result.stderr)

    result.check_returncode()

build_city_bundles(r"c:\Users\gabri\Documents\GitHub\referendum2026_mappepersezione")



Name                       Length
----                       ------
bologna.bundle.js         7332926
firenze.bundle.js         8145980
genova.bundle.js          7747527
milano.bundle.js         10849075
napoli.bundle.js         10217830
palermo.bundle.js        12996391
reggiocalabria.bundle.js 10156128
roma.bundle.js           37350966
taranto.bundle.js         6129600
torino.bundle.js          5708612





In [62]:
print("finish at:" +time.ctime())
end = time.ctime()

finish at:Mon Mar 23 21:42:50 2026


## Possibili aggiustamenti utili

- Se il click apre una nuova tab invece di un download diretto, dopo il `click()` controlla `driver.window_handles` e spostati sulla tab nuova.
- Se il bottone compare solo dopo aver scelto un livello territoriale, aggiungi i click intermedi prima della ricerca XPath.
- Se i file hanno nomi ambigui, rinominali con `citta + data` come gia' previsto sopra.
- Se alcune pagine falliscono, gli screenshot di debug vengono salvati in `downloads_csv/`.